# Phase 1: Data Exploration & Silver Layer Cleaning

In [2]:
import pandas as pd
import numpy as np
import glob
import os
import re
import unicodedata

# Options d'affichage Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("✅ Envirronement de travail prêt !")

✅ Envirronement de travail prêt !


## 1. ⚽ Exploration des Données SoFIFA (Bronze)

In [3]:
# Chargement du fichier SoFIFA Bronze
sofifa_files = glob.glob("../bronze_data/sofifa_pro_players_bronze_*.csv")
if not sofifa_files:
    sofifa_files = glob.glob("bronze_data/sofifa_pro_players_bronze_*.csv")

if sofifa_files:
    df_sofifa = pd.read_csv(sofifa_files[0])
    
    # 1. Dimensions
    print(f"📐 Shape SoFIFA : {df_sofifa.shape[0]} Lignes, {df_sofifa.shape[1]} Colonnes")
    
    # 2. Aperçu (Head)
    print("\n--- 5 Premières Lignes ---")
    display(df_sofifa.head())
    
    # 3. Informations & Types
    print("\n--- Informations sur les Types (info) ---")
    df_sofifa.info()
    
    # 4. Valeurs Manquantes
    print("\n--- Diagnostic des Valeurs Manquantes (Nulls) ---")
    nulls = df_sofifa.isnull().sum()
    print(nulls[nulls > 0] if not nulls[nulls > 0].empty else "✅ Aucune valeur nulle !")
    
    # 5. Doublons
    print(f"\n👯 Nombre de Doublons : {df_sofifa.duplicated().sum()}")
else:
    print("⚠️ Aucun fichier SoFIFA trouvé dans bronze_data !")

📐 Shape SoFIFA : 2793 Lignes, 29 Colonnes

--- 5 Premières Lignes ---


,Unnamed: 0,Name,Age,Overall rating,Potential,Team & Contract,ID,Height,Weight,foot,Value,Wage,Acceleration,Sprint speed,Stamina,Strength,Interceptions,Vision,Composure,Standing tackle,Sliding tackle,GK Diving,GK Handling,GK Positioning,Weak foot,Skill moves,Attacking work rate,Defensive work rate,Unnamed: 28
0,NaN,E. Haaland ST,24,91,93,Manchester City 2022 ~ 2034,239085,"195cm 6'5""",94kg 207lbs,Left,€172.5M,€390K,82,92,78,93,43,75,86,47,29,7,14,11,3,3,NaN,NaN,NaN
1,NaN,K. Mbappé ST LW LM,26,91,92,Real Madrid 2024 ~ 2029,231747,"182cm 6'0""",81kg 179lbs,Right,€157M,€610K,97,96,83,77,38,83,88,34,32,13,5,11,4,5,NaN,NaN,NaN
2,NaN,O. Dembélé ST RW CAM,28,90,90,Paris Saint-Germain 2023 ~ 2028,231443,"178cm 5'10""",67kg 148lbs,Left,€122.5M,€220K,93,89,76,69,45,84,88,49,39,6,6,10,5,5,NaN,NaN,NaN
3,NaN,Vitinha CM CDM CAM,25,90,92,Paris Saint-Germain 2022 ~ 2029,255253,"172cm 5'8""",64kg 141lbs,Right,€149M,€185K,75,69,89,59,85,91,88,79,69,12,13,5,3,4,NaN,NaN,NaN
4,NaN,H. Kane ST,31,90,90,FC Bayern München 2023 ~ 2027,202126,"188cm 6'2""",86kg 190lbs,Right,€101M,€170K,63,65,76,87,42,86,92,44,42,8,10,14,4,3,NaN,NaN,NaN



--- Informations sur les Types (info) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2793 entries, 0 to 2792
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           0 non-null      float64
 1   Name                 2793 non-null   object 
 2   Age                  2793 non-null   int64  
 3   Overall rating       2793 non-null   int64  
 4   Potential            2793 non-null   int64  
 5   Team & Contract      2793 non-null   object 
 6   ID                   2793 non-null   int64  
 7   Height               2793 non-null   object 
 8   Weight               2793 non-null   object 
 9   foot                 2793 non-null   object 
 10  Value                2793 non-null   object 
 11  Wage                 2793 non-null   object 
 12  Acceleration         2793 non-null   int64  
 13  Sprint speed         2793 non-null   int64  
 14  Stamina              2793 non-null   int64  


## 2. 🏆 Exploration de Tous les Fichiers de Ligues (Understat)

In [4]:
# Liste des fichiers de Ligues
league_files = ['premier_league.csv', 'la_liga.csv', 'bundesliga.csv', 'serie_a.csv', 'Ligue_1.csv']

print("==================================================")
print("📊 EXPLORATION DES FICHIERS UNDERSTAT (BRONZE)")
print("==================================================")

for f in league_files:
    # Test des deux chemins possibles (selon l'emplacement du notebook)
    path = os.path.join("../bronze_data", f)
    if not os.path.exists(path):
        path = os.path.join("bronze_data", f)
        
    if os.path.exists(path):
        print(f"\n📁 Fichier : {f}")
        print("-" * 40)
        
        # Lecture flexible pour éviter les parser errors
        df_l = pd.read_csv(path, sep=None, engine='python', on_bad_lines='skip')
        
        # 1. Dimensions
        print(f"  • Shape       : {df_l.shape[0]} Lignes | {df_l.shape[1]} Colonnes")
        
        # 2. Annee / Saison
        year_col = next((c for c in ['year', 'Year', 'season', 'Season'] if c in df_l.columns), None)
        years = df_l[year_col].unique() if year_col else "N/A"
        print(f"  • Années/Saisons: {years}")
        
        # 3. Doublons
        print(f"  • Doublons    : {df_l.duplicated().sum()}")
        
        # 4. Valeurs manquantes
        nulls_l = df_l.isnull().sum()
        nulls_l_filtered = nulls_l[nulls_l > 0]
        if not nulls_l_filtered.empty:
            print(f"  • Colonnes avec Nuls : {list(nulls_l_filtered.index)}")
        else:
            print("  • Valeurs Nulles     : 0")
            
        print(f"  • Aperçu colonnes    : {list(df_l.columns[:5])}")
    else:
        print(f"\n⚠️ Fichier non trouvé : {f}")

📊 EXPLORATION DES FICHIERS UNDERSTAT (BRONZE)

📁 Fichier : premier_league.csv
----------------------------------------
  • Shape       : 537 Lignes | 8 Colonnes
  • Années/Saisons: N/A
  • Doublons    : 0
  • Valeurs Nulles     : 0
  • Aperçu colonnes    : ['\ufeff"number"', 'player', 'team', 'apps', 'min']

📁 Fichier : la_liga.csv
----------------------------------------
  • Shape       : 600 Lignes | 8 Colonnes
  • Années/Saisons: N/A
  • Doublons    : 0
  • Valeurs Nulles     : 0
  • Aperçu colonnes    : ['\ufeff"number"', 'player', 'team', 'apps', 'min']

📁 Fichier : bundesliga.csv
----------------------------------------
  • Shape       : 499 Lignes | 8 Colonnes
  • Années/Saisons: N/A
  • Doublons    : 0
  • Valeurs Nulles     : 0
  • Aperçu colonnes    : ['\ufeff"number"', 'player', 'team', 'apps', 'min']

📁 Fichier : serie_a.csv
----------------------------------------
  • Shape       : 586 Lignes | 8 Colonnes
  • Années/Saisons: N/A
  • Doublons    : 0
  • Valeurs Nulles     :

## 3. 💡 Constats & Décisions pour la Couche Silver
- **Parsing des types:** Les colonnes financières (`Value`, `Wage`) et physiques (`Height`, `Weight`) doivent être nettoyées et converties en format numérique.
- **Séparation des colonnes combinées:** Découpage de `Name` (Nom + Position) et `Team & Contract` via Regex.
- **Colonnes Inutiles:** Suppression des colonnes vides ou indésirables (`Unnamed`, `Attacking work rate`).
- **Normalisation pour Fusion:** Création de la clé `Match_Name` (sans accents) pour réussir le `pd.merge()` entre SoFIFA et Understat.